<a href="https://colab.research.google.com/github/LeonardooAlves/LSE-ST445/blob/main/Week%208/1_st445_week8_lab1_matrix_tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ST445 Week 8 · Lab 1: visualising matrices, step by step

**A guided tutorial · about 90 minutes · follows the Week 8 lecture · no exercises: read each explanation, run each cell, and study the output**

The Week 8 lecture showed that a matrix display can reveal structure that a list of numbers hides, but only if it is ordered well. This notebook applies every technique from the lecture to one real dataset, in the order the lecture introduced them:

| Section | Technique | The question it answers |
|---|---|---|
| 1 | Loading and preprocessing (one cell) | Are the data ready for matrix displays? |
| 2 | **Pair plot** | What does each pair of variables look like? |
| 3 | **Correlation heatmap** | How strongly is every pair related, at once? |
| 4 | **Dot matrix** | The same matrix with a different encoding: what changes? |
| 5 | **Image of the data matrix** | What does the whole dataset look like, one cell per value? |
| 6 | **Robinson form** | How far is the current order from the ideal order? |
| 7 | **Seriation** | Which order of the variables reveals their structure? |
| 8 | **Clustered heatmaps** | Which groups of variables and of wines go together? |
| 9 | **Spectral seriation** | Can one eigenvector order 178 wines? |
| 10 | Insight statements | What can we claim, with what evidence, and with which caveats? |

Each technique has a Markdown cell **before** the code (what it is, and why we use it) and **after** it (how to read the output, and what it shows).

**Before you start:** choose **File → Save a copy in Drive**, so that your changes are saved.

---
## 0. The dataset: chemical analysis of Italian wines

The **Wine** dataset records 13 chemical measurements for 178 wines grown in the same region of Italy, from **three cultivars** (grape varieties). It is small enough to see every value, has enough variables for matrix displays to be useful, and has a known grouping (the cultivar) that we can use to **check** what the displays reveal, without using it to create them.

* **Source:** Forina, M. et al., PARVUS: an extendible package for data exploration, classification and correlation, Institute of Pharmaceutical and Food Analysis and Technologies, Genoa; distributed by the [UCI Machine Learning Repository, dataset 109](https://archive.ics.uci.edu/dataset/109/wine) under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).
* **Access:** a copy is included in scikit-learn, [`sklearn.datasets.load_wine`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_wine.html), so it loads without a download.
* **One row:** one wine. **Columns:** 13 measurements (alcohol, malic acid, ash, alkalinity of ash, magnesium, total phenols, flavanoids, non-flavanoid phenols, proanthocyanins, colour intensity, hue, the OD280/OD315 ratio of diluted wines, and proline), plus the cultivar (0, 1 or 2).

---
## 1. Set-up, loading and preprocessing

### Imports and style

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_wine
from scipy.cluster.hierarchy import linkage, optimal_leaf_ordering, leaves_list
from scipy.spatial.distance import squareform, pdist
from sklearn.neighbors import kneighbors_graph

# The Okabe-Ito colour-blind-safe palette (Week 7)
OKABE_ITO = ["#0072B2", "#E69F00", "#009E73", "#CC79A7", "#56B4E9", "#D55E00", "#F0E442", "#000000"]
CULTIVAR_COLOURS = {"cultivar 0": OKABE_ITO[0], "cultivar 1": OKABE_ITO[1], "cultivar 2": OKABE_ITO[2]}

#Changing the default parameters of matplotlib and seaborn
sns.set_theme(style="ticks")
plt.rcParams.update({
    "figure.dpi": 110,               # resolution on screen
    "axes.spines.top": False,        # remove the top border of every plot
    "axes.spines.right": False,      # remove the right border of every plot
})

### Preprocessing, in one cell

Everything needed to prepare the data is in the next cell, with a comment on what each step does and why. Two versions of the measurements are kept:

* **`wine`**, in the original units, for charts where the units matter (the pair plot);
* **`Z`**, **standardised** (each column shifted to mean 0 and scaled to standard deviation 1), for distances between wines. Without standardising, proline (values in the hundreds) would dominate every distance, and magnesium (around 100) would outweigh hue (around 1): the Week 8 lecture's covariance-versus-correlation point, applied to distances.

The cultivar is kept in a separate column. It is **never used to build a display**, only to check afterwards what a display has found.

In [ ]:
# 🟢 Load and prepare the data: one documented cell
raw = load_wine(as_frame=True).frame            # a DataFrame: 13 measurements + 'target' (the cultivar)
wine = raw.copy()                               # keep raw unchanged, and apply preprocessing onto wine dataframe
wine.head()                                     # peak into the data

In [ ]:
# 🟢 Prepare/Preprocess the data: one documented cell

# 1. Readable column names, so that they can be used directly as axis labels
wine = wine.rename(columns={
    "alcalinity_of_ash": "alkalinity_of_ash",               # the standard spelling
    "od280/od315_of_diluted_wines": "od280_od315",          # shorter; '/' is awkward in names
    "color_intensity": "colour_intensity",                  # British spelling, if you prefer
    "target": "cultivar",
})

# The target column, renamed to cultivar, is the wine category. It has three unique values: 0, 1, 2. So,
# 2. The cultivar as a labelled category, used only to check displays afterwards
wine["cultivar"] = pd.Categorical("cultivar " + wine["cultivar"].astype(str),
                                  categories=["cultivar 0", "cultivar 1", "cultivar 2"])

# 3. Shuffle the rows with a fixed seed. The published file is already sorted by cultivar, which would
#    give every display a 'free' good order; To make this data more similar to real messy data, we
#    shuffle it to make its order uninformative, as in real data
wine = wine.sample(frac=1, random_state=445).reset_index(drop=True)

# 4. The 13 measurements, and a standardised copy for distances (mean 0, standard deviation 1)
FEATURES = [c for c in wine.columns if c != "cultivar"] # copy all columns but cultivar, the target column.
Z = (wine[FEATURES] - wine[FEATURES].mean()) / wine[FEATURES].std()

# 5. Checks that fail loudly (Week 4)
assert wine.shape == (178, 14), f"unexpected shape {wine.shape}" # check the number of rows and columns are correct, otherwise raise an error
assert wine[FEATURES].notna().all().all(), "a missing value" # check if missing values exist
assert np.allclose(Z.mean(), 0) and np.allclose(Z.std(), 1), "standardisation failed" # check that all features were standardised with z-score

print(wine.shape)
print(wine["cultivar"].value_counts().sort_index().to_dict())
wine.head()

**What you should see:** `(178, 14)`, then 59, 71 and 48 wines in cultivars 0, 1 and 2, and the first five rows in a random order. There are no missing values, so no imputation is needed.

---
## 2. Pair plot: every pair of a few variables

A [pair plot](https://seaborn.pydata.org/generated/seaborn.pairplot.html) draws a grid of scatter plots, one for each pair of variables, with each variable's distribution on the diagonal. It is the matrix version of Week 7's protocol step 4 (relationships): one panel per cell of the correlation matrix.

**Why use it.** It shows the **shape** of each relationship (curves, clusters, outliers), which a correlation coefficient hides (Anscombe's quartet). Its limit is size: with p variables it has p × p panels, so 13 variables would give 169 small panels. We therefore choose five variables, and colour the points by cultivar **only to check** whether the structure we see corresponds to something known.

In [ ]:
# 🟢 A pair plot of five variables, coloured by cultivar to check what the structure corresponds to
five = ["alcohol", "flavanoids", "colour_intensity", "proline", "od280_od315"]

g = sns.pairplot(wine, vars=five,
                 hue="cultivar", palette=CULTIVAR_COLOURS,
                 diag_kind="kde",                      # a smooth density on the diagonal
                 plot_kws={"alpha": 0.5, "s": 18},     # transparent, small points against overplotting
                 height=1.9)
g.figure.suptitle("Five measurements of 178 wines, by cultivar", y=1.02, fontweight="bold")
plt.show()

**How to read it.** Each panel off the diagonal is a scatter plot of the row variable (vertical) against the column variable (horizontal); each panel on the diagonal is that variable's density, one curve per cultivar. The grid is symmetric: the panel below the diagonal is the panel above it, mirrored.

**What it shows.**

* **Flavanoids and OD280/OD315** rise together almost linearly: a strong positive relationship, visible as a tight diagonal cloud.
* **The three cultivars separate** in several panels: for example, in flavanoids against colour intensity, cultivar 2 sits at low flavanoids and high colour intensity, cultivar 0 at high flavanoids, and cultivar 1 at low colour intensity.
* **Proline** separates cultivar 0 (high values, around 1,100) from the other two; its distribution is right-skewed.

**Insight.** *Observation:* a few measurements separate the cultivars, and several measurements move together. *Evidence:* the pair plot of five variables. *Caveat:* five of 13 variables only; the colouring uses known labels, which an exploratory analysis of new data would not have. To see **all** pairs at once, we need a correlation matrix.

---
## 3. Correlation heatmap: every pair, in one display

A correlation matrix holds the Pearson correlation of every pair of variables (from −1 to +1). A [heatmap](https://seaborn.pydata.org/generated/seaborn.heatmap.html) shows it with one coloured cell per pair.

**Why use it, and how.** It summarises all 78 pairs of the 13 variables at once. Following the lecture and Week 7:

* a **diverging** colour map (`RdBu_r`), **centred on 0**, with limits fixed at −1 and 1, because correlation has a meaningful midpoint;
* values **written in the cells**, because colour is the least accurate visual channel;
* the **upper triangle hidden**, because it repeats the lower one (data-ink).

The variables are shown in the **order of the proprocessed file**: an arbitrary order. Look at how much structure is visible in it, then compare with section 7.

In [ ]:
# 🟢 The correlation matrix, in the file's preproceed (arbitrary) column order.
corr = wine[FEATURES].corr()                                 # 13 x 13 Pearson correlations

# np.triu() is used here to create a mask for a correlation heatmap so that the upper-triangular part (above the diagonal) is hidden.
# This prevents redundant information from being displayed, as correlation matrices are symmetric.
# np.ones_like(corr, dtype=bool): Creates a matrix of the same shape as the corr matrix, but fills it entirely with True boolean values.
# np.triu(..., k=1): triu stands for triangle upper, and k=1 keeps the main diagonal.
# When you pass this mask into sns.heatmap(..., mask=mask), Seaborn hides any cells where the mask is True.
# This leaves only the lower-triangular portion of the correlation heatmap visible, which reduces visual clutter.
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)          # True above the diagonal: those cells are hidden

fig, ax = plt.subplots(figsize=(9, 7.5))
sns.heatmap(corr, mask=mask,
            annot=True, fmt=".2f", annot_kws={"size": 8},    # each value, two decima places
            cmap="RdBu_r", vmin=-1, vmax=1, center=0,        # diverging, centred on zero
            square=True, linewidths=0.5,
            cbar_kws={"label": "Pearson correlation", "shrink": 0.8},
            ax=ax)
ax.set_title("Correlations between 13 measurements, in the file's order")
plt.show()

# The strongest pairs, for the interpretation below
pairs = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack().dropna().sort_values()   # each pair once
print("most negative:", pairs.head(3).round(2).to_dict())
print("most positive:", pairs.tail(3).round(2).to_dict())

**How to read it.** Each cell is the correlation between its row variable and its column variable: dark red is strongly positive, dark blue strongly negative, white close to zero.

**What it shows.**

* The **strongest relationship** is between total phenols and flavanoids (r = 0.86): flavanoids are one kind of phenol, so this is partly built in. Flavanoids and OD280/OD315 (0.79) and total phenols and OD280/OD315 (0.70) are next: a **group of phenol-related measurements** that rise together.
* The **strongest negative** correlations, around −0.5 to −0.56, link hue and malic acid, flavanoids and non-flavanoid phenols, and hue and colour intensity.
* In this order, the red and blue cells are **scattered**: you have to search cell by cell to find which variables belong together. That is the problem seriation solves (section 7).

**Caveat.** Pearson correlation measures straight-line association only; the pair plot in section 2 showed that some relationships are clustered rather than linear.

---
## 4. Dot matrix: the same matrix, encoded by area

A dot matrix draws one dot per cell, with its **area** proportional to the size of the value and its **colour** showing the sign. It is the "dot matrix" from the lecture's four standard displays.

**Why look at it.** It is common in reports, and comparing it with the heatmap shows the lecture's perceptual ranking in practice: area is judged less accurately than position or length, but more easily than shade, so large correlations stand out quickly while small differences are hard to compare.

In [ ]:
# 🟢 A dot matrix of the correlations: area = |r|, colour = sign
n = len(FEATURES)
rows, cols = np.tril_indices(n, k=-1)                # the lower triangle, without the diagonal
values = corr.values[rows, cols]

fig, ax = plt.subplots(figsize=(8, 7))
ax.scatter(cols, rows,
           s=np.abs(values) * 900,                   # the area of each dot is proportional to |r|
           c=np.where(values > 0, OKABE_ITO[5], OKABE_ITO[0]),   # vermilion positive, blue negative
           alpha=0.8, edgecolor="white")
ax.set_xticks(range(n), FEATURES, rotation=90)
ax.set_yticks(range(n), FEATURES)
ax.invert_yaxis()                                    # first variable at the top, as in the heatmap
ax.set_title("Dot matrix: dot area shows |r|; vermilion positive, blue negative")
ax.set_aspect("equal")
plt.show()

**How to read it.** A large vermilion dot is a strong positive correlation; a large blue dot a strong negative one; a tiny dot a correlation near zero.

**What it shows, compared with the heatmap.** The strongest pairs (total phenols and flavanoids, flavanoids and OD280/OD315) are found at a glance, as large dots. But comparing a correlation of 0.45 with one of 0.55 is much harder than in the annotated heatmap, because the eye judges area poorly (Week 7: a circle whose radius doubles looks four times larger).

**Best-practice note.** If you use a dot matrix, scale the **area** (not the radius) to the value, and give the reader the numbers elsewhere. For most purposes the annotated heatmap of section 3 is clearer. Both displays share the same weakness: the arbitrary order of the variables.

---
## 5. The data matrix as an image

[`plt.imshow`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.imshow.html) draws a matrix as an **image**, one pixel-like cell per value. Here the matrix is the **data matrix** itself: 178 wines (rows) by 13 standardised measurements (columns). This is a **two-mode** matrix: its rows and columns are different things.

**Why use it.** It shows every one of the 2,314 values at once. `imshow` is the right tool for a matrix this tall: `sns.heatmap` would draw 2,314 separate shapes, while `imshow` draws one image quickly; `interpolation="none"` keeps each cell sharp. We use a diverging colour map because standardised values have a meaningful centre, 0 (the mean).

We draw it twice: rows in the shuffled order, and rows sorted by cultivar, to see what a good row order can reveal.

In [ ]:
# 🟢 The standardised data matrix, in the shuffled order and sorted by cultivar
order_by_cultivar = np.argsort(wine["cultivar"].cat.codes.values, kind="stable")

fig, axs = plt.subplots(1, 2, figsize=(12, 6.5), layout="constrained")
for ax, rows_order, title in [(axs[0], np.arange(len(Z)), "Rows in a random order"),
                              (axs[1], order_by_cultivar, "Rows sorted by cultivar (for checking only)")]:
    im = ax.imshow(Z.values[rows_order],
                   cmap="RdBu_r", vmin=-3, vmax=3,        # standardised values: ±3 sd covers almost all
                   aspect="auto",                          # stretch cells to fill the axes
                   interpolation="none")                   # sharp cells, no blurring
    ax.set_xticks(range(len(FEATURES)), FEATURES, rotation=90)
    ax.set_ylabel("wines (rows)")
    ax.set_title(title)

# Mark where each cultivar starts in the sorted panel
bounds = np.cumsum(wine["cultivar"].value_counts().sort_index().values)[:-1]
for b in bounds:
    axs[1].axhline(b - 0.5, color="black", linewidth=1.5)
fig.colorbar(im, ax=axs, shrink=0.6, label="standardised value (sd from the mean)")
plt.show()

**How to read it.** Each row is one wine; each column one measurement; red means above that measurement's average, blue below.

**What it shows.**

* **Left:** in a random order, each column looks like noise: there is no visible structure in the rows.
* **Right:** sorted by cultivar, clear **bands** appear. Cultivar 0 (top) is red in alcohol, flavanoids, total phenols and proline; cultivar 1 (middle) is blue in alcohol and colour intensity; cultivar 2 (bottom) is red in colour intensity and malic acid, and blue in flavanoids, hue and OD280/OD315.

**Insight.** The **same values** look random or highly structured depending only on the **order of the rows**: the lecture's opening point, on real data. Sorting by a known label is cheating for exploration, though. Sections 6 to 9 find good orders **without** using the cultivar.

---
## 6. Robinson form: measuring how good an order is

A dissimilarity matrix is in **anti-Robinson form** when its values never decrease as you move away from the diagonal, along any row or column (Robinson, 1951): similar objects sit next to each other, and the most different ones far apart. Real data are rarely exactly in this form, so the lecture measured **how far** an order is from it, with two losses:

1. **Gradient-condition violations** (anti-Robinson events): for every triple i < k < j, one violation if d_ik > d_ij (within rows) and one if d_kj > d_ij (within columns). Fewer is better.
2. **Path length**: the sum of dissimilarities between neighbours in the order, d₁₂ + d₂₃ + .... Shorter is better.

**Why use it.** A heatmap's order is a choice. These numbers let us compare orders objectively, instead of by eye.

**Dissimilarity between variables.** We use **d = 1 − r**: two perfectly correlated variables have dissimilarity 0, uncorrelated ones 1, and perfectly negatively correlated ones 2.

In [ ]:
# 🟢 The two loss functions from the lecture
def ar_events(D):
    """Loss 1: the number of violations of the gradient condition (anti-Robinson events)."""
    D = np.asarray(D)
    n = len(D)
    count = 0
    for i in range(n):
        for j in range(i + 2, n):
            k = np.arange(i + 1, j)                       # every k strictly between i and j
            count += int((D[i, k] > D[i, j]).sum())       # within row i
            count += int((D[k, j] > D[i, j]).sum())       # within column j
    return count

def path_length(D, order):
    """Loss 2: the sum of dissimilarities between neighbours in the order."""
    order = list(order)
    return float(sum(D[a, b] for a, b in zip(order[:-1], order[1:])))

def n_comparisons(n):
    """The number of comparisons the gradient condition makes: 2 per triple."""
    return 2 * n * (n - 1) * (n - 2) // 6

D_var = 1 - corr.values                                    # dissimilarity between variables
original = list(range(len(FEATURES)))

print("violations in the file's order:", ar_events(D_var), "of", n_comparisons(13), "comparisons")
print("path length in the file's order:", round(path_length(D_var, original), 2))

**How to read it.** The file's order of the 13 variables violates the gradient condition 283 times out of 572 comparisons (about half: no better than chance), and its path length is about 10.9. These are the numbers to beat.

**Why we cannot simply try every order.** There are 13! ≈ 6.2 billion orders of 13 variables. Checking them all, as we did for the lecture's 4 × 4 example, is not practical; we need a method that finds a good order directly (section 7).

---
## 7. Seriation: reordering the variables

We use the method the lecture recommended for real data: **hierarchical clustering** with average linkage, whose dendrogram gives an order of the leaves, followed by **optimal leaf ordering** (Bar-Joseph et al., 2001), which flips the branches of the tree so that the most similar neighbours sit side by side. In SciPy:

* [`squareform`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.spatial.distance.squareform.html) turns the square dissimilarity matrix into the **condensed** form that `linkage` expects (its upper triangle as one vector); `checks=False` accepts a diagonal that is zero only up to rounding;
* [`linkage`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.cluster.hierarchy.linkage.html) builds the tree;
* [`optimal_leaf_ordering`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.cluster.hierarchy.optimal_leaf_ordering.html) chooses the best flips;
* [`leaves_list`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.cluster.hierarchy.leaves_list.html) returns the resulting order.

**Why use it.** It is fast for any number of objects, and it gives an order that respects the groups the tree finds.

In [ ]:
# 🟢 Seriate the variables: average-linkage tree, then optimal leaf ordering
condensed = squareform(D_var, checks=False)               # the upper triangle of D_var as one vector
tree = linkage(condensed, method="average")
tree_olo = optimal_leaf_ordering(tree, condensed)
order = leaves_list(tree_olo)                              # the new order of the 13 variables

print("new order:", [FEATURES[i] for i in order])
print("violations: before", ar_events(D_var), "| after", ar_events(D_var[np.ix_(order, order)]))
print("path length: before", round(path_length(D_var, original), 2), "| after", round(path_length(D_var, order), 2))

**What the numbers show.** Seriation cuts the violations from 283 to 74 (from about half of the comparisons to about 13%) and the path length from about 10.9 to 6.0. The new order puts malic acid, non-flavanoid phenols and alkalinity of ash at one end, and the phenol group (proanthocyanins, total phenols, flavanoids, OD280/OD315) and hue at the other.

Now draw the correlation heatmap again, in the new order.

In [ ]:
# 🟢 The correlation heatmap, before and after seriation
ordered_names = [FEATURES[i] for i in order]

fig, axs = plt.subplots(1, 2, figsize=(15, 6.5), layout="constrained")
for ax, names, title in [(axs[0], FEATURES, "The file's order: 283 violations"),
                         (axs[1], ordered_names, "Seriated order: 74 violations")]:
    sns.heatmap(corr.loc[names, names],
                cmap="RdBu_r", vmin=-1, vmax=1, center=0, square=True,
                cbar=ax is axs[1], cbar_kws={"label": "Pearson correlation", "shrink": 0.7},
                ax=ax)
    ax.set_title(title)
plt.show()

**How to read it.** The two heatmaps contain exactly the same 169 numbers; only the order of the rows and columns differs.

**What it shows.** In the seriated order the matrix shows **two blocks of mutually positive correlations** (red squares on the diagonal), with **negative correlations between them** (blue rectangles off the diagonal):

* a **phenol block**: total phenols, flavanoids, proanthocyanins, OD280/OD315 and hue all rise together;
* a smaller block of malic acid, non-flavanoid phenols and alkalinity of ash, which tend to be high when the phenol block is low;
* alcohol, proline, colour intensity and magnesium sit between them, related more weakly to both.

**Insight.** *Observation:* the 13 measurements form two opposing groups, phenol-related measurements and an acidity-related group, with the others in between. *Evidence:* the seriated correlation heatmap; violations reduced from 283 to 74. *Caveat:* the order depends on the dissimilarity (1 − r) and the linkage (average); a different choice could move the variables in the middle.

---
## 8. Clustered heatmaps

A [clustered heatmap](https://seaborn.pydata.org/generated/seaborn.clustermap.html) combines the two previous steps: it reorders rows and columns by a hierarchical clustering and draws the **dendrograms** beside the matrix, so the reader sees both the order and the tree that produced it (Eisen et al., 1998).

**Why use it, and the key detail.** It shows which variables join at low heights (strongly related) and which only at the top. As the lecture stressed, `sns.clustermap` treats each **row** of its input as an observation. Our correlation matrix is not a list of observations, so we compute the linkage from the dissimilarities 1 − r ourselves and pass it with `row_linkage` and `col_linkage`.

### 8.1 The variables

In [ ]:
# 🟢 A clustered heatmap of the correlations, using the linkage computed from 1 - r
g = sns.clustermap(corr,
                   row_linkage=tree_olo, col_linkage=tree_olo,   # the tree from section 7, with optimal leaf order
                   cmap="RdBu_r", vmin=-1, vmax=1, center=0,
                   annot=True, fmt=".1f", annot_kws={"size": 7},
                   figsize=(9, 8),
                   dendrogram_ratio=0.15,                         # the share of the figure for the trees
                   cbar_pos=(0.02, 0.85, 0.03, 0.12),
                   cbar_kws={"label": "r"})
g.figure.suptitle("Clustered correlation heatmap (average linkage on 1 − r)", y=1.02, fontweight="bold")
plt.show()

**How to read it.** The dendrogram's vertical axis is the merge height: the average dissimilarity (1 − r) at which two branches join. Variables joined near the bottom are strongly correlated; branches that join only at the top are weakly or negatively related.

**What it shows.** Total phenols and flavanoids join first (1 − r ≈ 0.14, that is r ≈ 0.86), then OD280/OD315 and proanthocyanins join them: the phenol block from section 7. The tree also shows **how** the blocks relate: the acidity-related group joins the rest only near the top.

### 8.2 The wines: a two-mode clustered heatmap

Now the **data matrix** itself (178 wines × 13 standardised measurements). Here each row really is an observation, so `clustermap` can compute the distances itself: Euclidean distances between wines, and between measurements. A colour strip on the left shows each wine's cultivar, **added after clustering** to check the result.

In [ ]:
# 🟢 Cluster the wines and the measurements at the same time; show the cultivar only as a check
row_colours = wine["cultivar"].map(CULTIVAR_COLOURS).astype(str)     # one colour per wine, for the side strip

g = sns.clustermap(Z.set_axis(range(len(Z))),
                   method="ward", metric="euclidean",          # Ward linkage: compact, similar-sized groups
                   cmap="RdBu_r", vmin=-3, vmax=3, center=0,
                   row_colors=row_colours.values,              # the check: cultivar beside each row
                   yticklabels=False,                          # 178 wine labels would be unreadable
                   figsize=(9, 10), dendrogram_ratio=(0.15, 0.1),
                   cbar_pos=(0.02, 0.85, 0.03, 0.1), cbar_kws={"label": "sd"})
g.figure.suptitle("178 wines × 13 measurements, clustered (Ward, Euclidean)", y=1.01, fontweight="bold")
plt.show()

# How well do the three top-level branches of the wine tree match the cultivars?
from scipy.cluster.hierarchy import fcluster
groups = fcluster(g.dendrogram_row.linkage, t=3, criterion="maxclust")   # cut the tree into 3 groups
pd.crosstab(pd.Series(groups, name="cluster"), wine["cultivar"])

**How to read it.** Rows are wines, columns measurements, both reordered by their own trees. The coloured strip on the left shows the cultivar of each wine, which the clustering did **not** use.

**What it shows.** The wines fall into **three large branches**, and the colour strip shows that each branch is mostly one cultivar. The cross-table counts this: cutting the tree into three groups puts all 59 wines of cultivar 0 in one group and all 48 of cultivar 2 in another, while 13 of the 71 wines of cultivar 1 (8 + 5) join one of those two groups: 165 of 178 wines (93%) are grouped with their own cultivar. The column tree reproduces the variable groups of section 8.1, and the blocks of colour show **which measurements characterise each group of wines**: high phenols, alcohol and proline for one, high colour intensity and malic acid with low phenols for another, low alcohol and colour intensity for the third.

**Insight.** *Observation:* without being told the cultivars, hierarchical clustering of the 13 measurements separates the wines into three groups that largely match them. *Evidence:* the two-mode clustered heatmap and the cross-table. *Caveat:* Ward linkage on standardised data was chosen; other linkages and scalings give different trees, and three groups were chosen because three cultivars are known, which real exploration would not know (Week 8: k is chosen, not discovered).

---
## 9. Spectral seriation of the 178 wines

The lecture's last ordering method uses one **eigenvector** instead of a search (Atkins et al., 1998):

1. build a **similarity** matrix S between the wines;
2. compute the **graph Laplacian** L = D − S, where D is the diagonal matrix of each wine's total similarity;
3. take the eigenvector of L with the **second-smallest** eigenvalue, the **Fiedler vector**: one number per wine;
4. **sort** the wines by it.

The intuition from the lecture: the Fiedler vector places each wine on a line so that strongly similar wines are close, by minimising Σ Sᵢⱼ (xᵢ − xⱼ)².

**Why use it.** It orders all 178 wines with one eigen-decomposition, with no search over 178! orders, and no number of groups to choose.

**The similarity.** We connect each wine to its **10 nearest neighbours** in the standardised measurements ([`kneighbors_graph`](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.kneighbors_graph.html)), giving similarity 1 between neighbours and 0 otherwise, and make the matrix symmetric. A neighbourhood graph like this is a standard choice for spectral methods (von Luxburg, 2007): it keeps only the local, reliable similarities.

In [ ]:
# 🟢 Spectral seriation of the wines
# 1. Similarity: each wine linked to its 10 nearest neighbours (in standardised units)
A = kneighbors_graph(Z.values, n_neighbors=10, mode="connectivity").toarray()   # 1 if j is among i's neighbours
S = (A + A.T) / 2                                                               # symmetric: average the two directions

# 2. The graph Laplacian
L = np.diag(S.sum(axis=1)) - S

# 3. Eigen-decomposition; eigh returns eigenvalues in increasing order
eigenvalues, eigenvectors = np.linalg.eigh(L)
fiedler = eigenvectors[:, 1]                       # the eigenvector of the second-smallest eigenvalue

# 4. The order: wines sorted by their Fiedler value
spectral_order = np.argsort(fiedler)

print("smallest eigenvalues:", eigenvalues[:3].round(4))

**What you should see:** the smallest eigenvalue is 0 (always true for a Laplacian), and the second is positive: the graph is connected, so the Fiedler vector is meaningful.

Now compare three orders of the 178 × 178 **distance matrix** between wines: the random order, the spectral order, and, as a reference, the order sorted by cultivar.

In [ ]:
# 🟢 The distance matrix between wines in three orders, with the gradient-condition loss of each
D_wine = squareform(pdist(Z.values))               # Euclidean distances between standardised wines

orders = {
    "random order": np.arange(len(Z)),
    "spectral (Fiedler) order": spectral_order,
    "sorted by cultivar (reference)": order_by_cultivar,
}

fig, axs = plt.subplots(1, 3, figsize=(16, 5.8), layout="constrained")
for ax, (label, o) in zip(axs, orders.items()):
    violations = ar_events(D_wine[np.ix_(o, o)])
    im = ax.imshow(D_wine[np.ix_(o, o)],
                   cmap="magma",                                  # dark = similar, bright = different
                   vmin=0, vmax=np.percentile(D_wine, 75),        # clip the largest distances for contrast
                   interpolation="none")
    ax.set_title(f"{label}\n{violations:,} violations", fontsize=11)
    ax.set_xticks([])
    ax.set_yticks([])
fig.colorbar(im, ax=axs, shrink=0.7, label="Euclidean distance (standardised units; top quarter clipped)")
plt.show()
print("comparisons in total:", f"{n_comparisons(178):,}")

**How to read it.** Each image is the same 178 × 178 matrix of distances between wines; **dark cells are similar pairs** and bright cells very different ones (the largest quarter of distances is drawn in the brightest colour, to increase contrast). A good order puts dark cells along the diagonal and bright cells in the corners: the anti-Robinson pattern.

**What it shows.** The random order looks like noise (about 931,000 violations out of 1,848,352 comparisons, roughly half). The spectral order produces a clear structure, dark along the diagonal and bright in the corners, with about 401,000 violations: **fewer even than the order sorted by cultivar** (about 586,000), because it also orders the wines **within** each cultivar, so that neighbours in the order are similar. The last cell shows where the cultivars fall along the Fiedler vector.

In [ ]:
# 🟢 Each wine's Fiedler value, in spectral order, coloured by cultivar (added only as a check)
fig, ax = plt.subplots(figsize=(12, 3.6))
ordered = wine.iloc[spectral_order]
ax.bar(range(len(ordered)), fiedler[spectral_order],
       color=ordered["cultivar"].map(CULTIVAR_COLOURS),
       width=1.0)
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in CULTIVAR_COLOURS.values()]
ax.legend(handles, CULTIVAR_COLOURS.keys(), loc="upper left", frameon=False)
ax.set(title="The Fiedler vector orders the wines along a gradient from one cultivar to another",
       xlabel="wines, in spectral order", ylabel="Fiedler value")
plt.show()

# Mean position of each cultivar along the spectral order (0 = first, 177 = last)
position = pd.Series(np.empty(len(Z)), index=range(len(Z)))
position.iloc[spectral_order] = np.arange(len(Z))
print(position.groupby(wine["cultivar"].values).mean().round(0))

**How to read it.** Each bar is one wine, in the spectral order, at the height of its Fiedler value; its colour is its cultivar, which the method did not use.

**What it shows.** The Fiedler vector arranges the wines along a **gradient**: mostly cultivar 2 at one end, cultivar 0 at the other, and cultivar 1 in between, with some mixing where cultivars overlap. The printed mean positions confirm it: on average, cultivar 2 is at position 24, cultivar 1 at 83 and cultivar 0 at 148 of the 178. (The sign of an eigenvector is arbitrary: if your bars run the other way, the order is simply reversed.)

**Insight.** *Observation:* one eigenvector, computed from the measurements alone, orders the wines from one cultivar through a second to the third. *Evidence:* the spectrally ordered distance matrix (about 401,000 violations, against about 931,000 in random order and about 586,000 sorted by cultivar) and the coloured Fiedler bars. *Caveat:* the result depends on the similarity graph (10 nearest neighbours, standardised measurements); spectral seriation finds a **single** gradient, so groups that do not lie along one line would be mixed. It is an exploratory order, not a classification.

---
## 10. Summary: what the matrix displays revealed

| Display | Observation | Evidence | Caveat |
|---|---|---|---|
| Pair plot | A few measurements separate the cultivars; flavanoids and OD280/OD315 rise together | Section 2 | Only 5 of 13 variables |
| Correlation heatmap | The strongest pair is total phenols and flavanoids (r = 0.86) | Section 3 | Linear association only; order arbitrary |
| Dot matrix | Large correlations stand out; small differences are hard to compare | Section 4 | Area is judged poorly |
| Data matrix image | The same values look random or banded depending on the row order | Section 5 | Sorting by cultivar uses known labels |
| Seriation | Two opposing groups of measurements: phenol-related and acidity-related | Section 7: violations 283 → 74 | Depends on 1 − r and average linkage |
| Clustered heatmap | Clustering the measurements groups 93% of wines with their own cultivar | Section 8.2, cross-table | Ward linkage; three groups chosen |
| Spectral seriation | One eigenvector orders the wines cultivar 2 → 1 → 0; violations about 931,000 → 401,000 | Section 9 | One gradient only; depends on the neighbourhood graph |

**The technique to remember.** Every display in this notebook showed the **same data**; the structure appeared only when the order was chosen well. Seriation, clustering and spectral methods are three ways to choose that order without knowing the answer in advance, and the loss functions of section 6 let you compare the orders they produce.

### References

* Atkins, J. E., Boman, E. G. and Hendrickson, B. (1998). A spectral algorithm for seriation and the consecutive ones problem. *SIAM Journal on Computing*, 28(1), 297–310.
* Bar-Joseph, Z., Gifford, D. K. and Jaakkola, T. S. (2001). Fast optimal leaf ordering for hierarchical clustering. *Bioinformatics*, 17(suppl. 1), S22–S29.
* Eisen, M. B., Spellman, P. T., Brown, P. O. and Botstein, D. (1998). Cluster analysis and display of genome-wide expression patterns. *PNAS*, 95(25), 14863–14868.
* Forina, M. et al. (1991). Wine dataset, UCI Machine Learning Repository, [doi:10.24432/C5PC7J](https://doi.org/10.24432/C5PC7J).
* Robinson, W. S. (1951). A method for chronologically ordering archaeological deposits. *American Antiquity*, 16(4), 293–301.
* von Luxburg, U. (2007). A tutorial on spectral clustering. *Statistics and Computing*, 17(4), 395–416.
* Documentation: [seaborn](https://seaborn.pydata.org/api.html), [SciPy hierarchical clustering](https://docs.scipy.org/doc/scipy/reference/cluster.hierarchy.html), [scikit-learn](https://scikit-learn.org/stable/api/index.html).